In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q datasets pandas numpy

import os
import glob
import pandas as pd
import numpy as np
import gc
from datasets import load_from_disk

                                
                                                                        
                                                                            
                                                                      
                                                                 
                                                                        
                                                                      
                                    
 
                                                                          
                                                                 
                                                                  

PROJECT_DIR = '/content/drive/MyDrive/man-bot-detection'
os.makedirs(f'{PROJECT_DIR}/data/processed', exist_ok=True)

print("Підключення виконано, бібліотеки завантажено.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Підключення виконано, бібліотеки завантажено.


In [ ]:
import os
import glob
import pandas as pd
import numpy as np

                 
PROJECT_DIR = '/content/drive/MyDrive/man-bot-detection'
DELBOT_PATH = f'{PROJECT_DIR}/data/raw/delbot_mouse'
OUTPUT_CSV = f'{PROJECT_DIR}/data/processed/behavioral_features_windowed.csv'

                                
def normalize_coords(traj):
    if len(traj) == 0:
        return traj
    min_x, max_x = np.min(traj[:,0]), np.max(traj[:,0])
    min_y, max_y = np.min(traj[:,1]), np.max(traj[:,1])
    range_x = max_x - min_x if max_x - min_x > 0 else 1
    range_y = max_y - min_y if max_y - min_y > 0 else 1
    norm_x = (traj[:,0] - min_x) / range_x * 200
    norm_y = (traj[:,1] - min_y) / range_y * 200
    return np.column_stack((norm_x, norm_y))

                                                     
def extract_window_features(window_coords, window_clicks=None):
    if len(window_coords) < 5:
        return None
    dx = np.diff(window_coords[:,0])
    dy = np.diff(window_coords[:,1])
    speeds = np.sqrt(dx**2 + dy**2)
    if len(speeds) == 0:
        return None

    mean_speed = np.mean(speeds)
    max_speed = np.max(speeds)
    std_speed = np.std(speeds)
    path_length = np.sum(speeds)
    num_stops = np.sum(speeds < 0.5)                                 

                           
    angles = np.arctan2(dy, dx)
    if len(angles) > 1:
        angle_diffs = np.abs(np.diff(angles))
        angle_diffs = np.minimum(angle_diffs, 2*np.pi - angle_diffs)
        mean_angle = np.mean(angle_diffs)
    else:
        mean_angle = 0

                                                   
    num_clicks = 0
    if window_clicks is not None and len(window_clicks) == len(window_coords):
                                           
        num_clicks = np.sum((window_clicks[1:] == 1) & (window_clicks[:-1] == 0))

    return {
        'mean_speed': mean_speed,
        'max_speed': max_speed,
        'std_speed': std_speed,
        'path_length': path_length,
        'num_stops': num_stops,
        'mean_angle': mean_angle,
        'num_clicks': num_clicks,
        'num_points': len(window_coords)
    }

                           
WINDOW_SIZE = 100                            
STEP = 50                                    

                       
human_files = glob.glob(f'{DELBOT_PATH}/circles_human_*/*.csv')
if not human_files:
    human_files = glob.glob(f'{DELBOT_PATH}/circles_human_*/*')
bot_files = glob.glob(f'{DELBOT_PATH}/circles_bot_*/*.csv')
if not bot_files:
    bot_files = glob.glob(f'{DELBOT_PATH}/circles_bot_*/*')

print(f"Знайдено людей: {len(human_files)}, ботів: {len(bot_files)}")

                                       
with open(OUTPUT_CSV, 'w') as f:
    f.write('mean_speed,max_speed,std_speed,path_length,num_stops,mean_angle,num_clicks,num_points,label,source\n')

                                                             
                                                                    
                                                                        
                                                                      
                                                                         
                                       
def load_delbot_trajectory(file_path):
    """Читає .txt-файл DELBOT-Mouse напряму, рядок за рядком.
    Повертає (coords, is_down) — масив (N,2) координат x,y та масив (N,)
    стану натискання миші (1 = кнопка затиснута, 0 = ні) на кожній точці."""
    coords = []
    is_down_flags = []
    is_down = 0
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('resolution'):
                continue
            parts = line.split(',')
            if len(parts) < 4:
                continue
            action = parts[1]
            try:
                x = float(parts[2])
                y = float(parts[3])
            except ValueError:
                continue
                                                                           
                                                                            
                                                                             
                                                                      
            if not (np.isfinite(x) and np.isfinite(y)):
                continue
            if action == 'Pressed':
                is_down = 1
            elif action == 'Released':
                is_down = 0
            coords.append((x, y))
            is_down_flags.append(is_down)
    return (np.array(coords, dtype=np.float32),
            np.array(is_down_flags, dtype=np.int32))

                                         
def process_files(file_list, label, source):
    count_windows = 0
    skipped_short = 0
    skipped_error = 0
    for file_path in file_list:
        try:
            coords, is_down = load_delbot_trajectory(file_path)
            if len(coords) < WINDOW_SIZE:
                skipped_short += 1
                continue
                          
            coords_norm = normalize_coords(coords)
            clicks = is_down
                               
            for start in range(0, len(coords_norm) - WINDOW_SIZE + 1, STEP):
                window_coords = coords_norm[start:start+WINDOW_SIZE]
                window_clicks = clicks[start:start+WINDOW_SIZE] if clicks is not None else None
                feats = extract_window_features(window_coords, window_clicks)
                if feats is None:
                    continue
                             
                with open(OUTPUT_CSV, 'a') as f:
                    f.write(f"{feats['mean_speed']:.4f},{feats['max_speed']:.4f},"
                            f"{feats['std_speed']:.4f},{feats['path_length']:.4f},"
                            f"{feats['num_stops']},{feats['mean_angle']:.4f},"
                            f"{feats['num_clicks']},{feats['num_points']},{label},{source}\n")
                count_windows += 1
        except Exception as e:
            skipped_error += 1
            continue
    print(f"   (файлів {label}: пропущено через коротку довжину (<{WINDOW_SIZE} точок) — "
          f"{skipped_short}, пропущено через помилку читання — {skipped_error})")
    return count_windows

                         
print("Обробка людей...")
human_windows = process_files(human_files, 'human', 'delbot_mouse_human')
print(f"Додано {human_windows} вікон від людей")

                         
print("Обробка ботів...")
bot_windows = process_files(bot_files, 'bot', 'delbot_mouse_bot')
print(f"Додано {bot_windows} вікон від ботів")
print(f"\n✅ Всього вікон: {human_windows + bot_windows}")
print(f"📁 Збережено у {OUTPUT_CSV}")

Знайдено людей: 955, ботів: 2596
Обробка людей...
   (файлів human: пропущено через коротку довжину (<100 точок) — 521, пропущено через помилку читання — 0)
Додано 1148 вікон від людей
Обробка ботів...
   (файлів bot: пропущено через коротку довжину (<100 точок) — 2200, пропущено через помилку читання — 0)
Додано 3165 вікон від ботів

✅ Всього вікон: 4313
📁 Збережено у /content/drive/MyDrive/man-bot-detection/data/processed/behavioral_features_windowed.csv


In [ ]:
                                                                            
                                                                               
                                                                            
                                                                              
                                                                       
                                                                       
                                                                        
                                                                        
                                                

CAPTCHA_CACHE_DIR = f'{PROJECT_DIR}/data/raw/captchasolve30k_hf_cache'

if not os.path.isdir(CAPTCHA_CACHE_DIR) or not os.listdir(CAPTCHA_CACHE_DIR):
    raise RuntimeError(
        "Кеш CaptchaSolve30k не знайдено на Drive за шляхом "
        f"{CAPTCHA_CACHE_DIR}. Спершу запусти комірку завантаження "
        "CaptchaSolve30k у 01_data_and_eda_FIXED.ipynb (вона якраз і "
        "створює цей кеш) — тоді цю комірку можна буде запустити без "
        "жодного звернення до HuggingFace/токена."
    )

print("Завантажуємо CaptchaSolve30k з кешу на Drive...")
captcha_dataset = load_from_disk(CAPTCHA_CACHE_DIR)

                                                                         
                                                                             
                                                                             
CAPTCHA_SPLIT = 'test'


def extract_coords_from_tickinputs(tick_inputs):
    """CaptchaSolve30k зберігає 'tickInputs' — список словників
    {'isDown', 'sampleIndex', 'x', 'y'}. Через дублювання по кадрах фізики
    багато послідовних записів мають однакові (x, y); прибираємо послідовні
    дублікати, щоб отримати чисту траєкторію руху (аналогічно до Move-подій
    у форматі DELBOT-Mouse)."""
    coords = []
    prev = None
    for p in tick_inputs:
        xy = (float(p['x']), float(p['y']))
        if xy != prev:
            coords.append(xy)
            prev = xy
    return np.array(coords, dtype=np.float32)


print(f"Обробка CaptchaSolve30k, спліт '{CAPTCHA_SPLIT}' "
      f"({len(captcha_dataset[CAPTCHA_SPLIT])} сесій)...")

captcha_windows = 0
for example in captcha_dataset[CAPTCHA_SPLIT]:
    coords = extract_coords_from_tickinputs(example['tickInputs'])
    if len(coords) < WINDOW_SIZE:
        continue
    coords_norm = normalize_coords(coords)
                                                                            
                                                                         
                                                                      
                                                                         
                      
    for start in range(0, len(coords_norm) - WINDOW_SIZE + 1, STEP):
        window_coords = coords_norm[start:start + WINDOW_SIZE]
        feats = extract_window_features(window_coords, None)
        if feats is None:
            continue
        with open(OUTPUT_CSV, 'a') as f:
            f.write(f"{feats['mean_speed']:.4f},{feats['max_speed']:.4f},"
                    f"{feats['std_speed']:.4f},{feats['path_length']:.4f},"
                    f"{feats['num_stops']},{feats['mean_angle']:.4f},"
                    f"{feats['num_clicks']},{feats['num_points']},"
                    f"human,captchasolve30k_human\n")
        captcha_windows += 1

print(f"Додано {captcha_windows} вікон від CaptchaSolve30k (людина)")
print(f"\n✅ Всього вікон (DELBOT + CaptchaSolve30k): "
      f"{human_windows + bot_windows + captcha_windows}")
print(f"📁 Збережено у {OUTPUT_CSV}")

Завантажуємо CaptchaSolve30k з кешу на Drive...
Обробка CaptchaSolve30k, спліт 'test' (5000 сесій)...
Додано 34911 вікон від CaptchaSolve30k (людина)

✅ Всього вікон (DELBOT + CaptchaSolve30k): 39224
📁 Збережено у /content/drive/MyDrive/man-bot-detection/data/processed/behavioral_features_windowed.csv


In [ ]:
from sklearn.model_selection import train_test_split

                                                   
df = pd.read_csv(f'{PROJECT_DIR}/data/processed/behavioral_features_windowed.csv')

                                                 
X = df.drop(['label', 'source'], axis=1)
           
y = df['label']

                                                             
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,                    
    stratify=y,                                        
    random_state=42                            
)

                    
print(f"Тренувальна вибірка: {X_train.shape[0]} рядків")
print(f"Тестова вибірка: {X_test.shape[0]} рядків")
print("\nРозподіл класів у тренувальній:")
print(y_train.value_counts(normalize=True))
print("\nРозподіл класів у тестовій:")
print(y_test.value_counts(normalize=True))

Тренувальна вибірка: 31379 рядків
Тестова вибірка: 7845 рядків

Розподіл класів у тренувальній:
label
human    0.919309
bot      0.080691
Name: proportion, dtype: float64

Розподіл класів у тестовій:
label
human    0.919312
bot      0.080688
Name: proportion, dtype: float64


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import os

                                                                 
                                  

          
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)
rf.fit(X_train, y_train)

        
y_pred = rf.predict(X_test)
print("="*50)
print("ЗВІТ КЛАСИФІКАЦІЇ (Random Forest, 300 дерев)")
print("="*50)
print(classification_report(y_test, y_pred))
print("\nМатриця плутанини:")
print(confusion_matrix(y_test, y_pred))
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")

                  
importances = rf.feature_importances_
feature_names = X.columns
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)
print("\nТоп-10 найважливіших ознак:")
print(feat_imp.head(10))

                                                           
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
joblib.dump(rf, f'{PROJECT_DIR}/models/behavioral_rf_300.pkl')
print(f"\n✅ Модель збережено у {PROJECT_DIR}/models/behavioral_rf_300.pkl")

ЗВІТ КЛАСИФІКАЦІЇ (Random Forest, 300 дерев)
              precision    recall  f1-score   support

         bot       0.93      0.83      0.88       633
       human       0.99      0.99      0.99      7212

    accuracy                           0.98      7845
   macro avg       0.96      0.91      0.93      7845
weighted avg       0.98      0.98      0.98      7845


Матриця плутанини:
[[ 524  109]
 [  39 7173]]

Accuracy: 0.9811

Топ-10 найважливіших ознак:
mean_angle     0.310704
num_stops      0.309639
std_speed      0.120837
max_speed      0.117436
mean_speed     0.071956
path_length    0.069266
num_clicks     0.000162
num_points     0.000000
dtype: float64

✅ Модель збережено у /content/drive/MyDrive/man-bot-detection/models/behavioral_rf_300.pkl


In [ ]:
                                                                        
                                                                           
                                                                         
                                                                          
                                                                         
                          

import os
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import gc

                 
PROJECT_DIR = '/content/drive/MyDrive/man-bot-detection'
DELBOT_PATH = f'{PROJECT_DIR}/data/raw/delbot_mouse'

                     
MAX_LEN = 100                                         

                                           
def normalize_coords(traj):
    if len(traj) == 0:
        return traj
    min_x, max_x = np.min(traj[:,0]), np.max(traj[:,0])
    min_y, max_y = np.min(traj[:,1]), np.max(traj[:,1])
    range_x = max_x - min_x if max_x - min_x > 0 else 1
    range_y = max_y - min_y if max_y - min_y > 0 else 1
    norm_x = (traj[:,0] - min_x) / range_x * 200
    norm_y = (traj[:,1] - min_y) / range_y * 200
    return np.column_stack((norm_x, norm_y))

                                                                          
                                                        
def load_delbot_trajectory(file_path):
    """Читає .txt-файл DELBOT-Mouse напряму, рядок за рядком.
    Повертає (coords, is_down) — масив (N,2) координат x,y та масив (N,)
    стану натискання миші (1 = кнопка затиснута, 0 = ні) на кожній точці."""
    coords = []
    is_down_flags = []
    is_down = 0
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('resolution'):
                continue
            parts = line.split(',')
            if len(parts) < 4:
                continue
            action = parts[1]
            try:
                x = float(parts[2])
                y = float(parts[3])
            except ValueError:
                continue
                                                                           
            if not (np.isfinite(x) and np.isfinite(y)):
                continue
            if action == 'Pressed':
                is_down = 1
            elif action == 'Released':
                is_down = 0
            coords.append((x, y))
            is_down_flags.append(is_down)
    return (np.array(coords, dtype=np.float32),
            np.array(is_down_flags, dtype=np.int32))

                                                                           
def load_trajectories(file_list, label, max_len=MAX_LEN):
    sequences = []
    labels = []
    skipped_short = 0
    skipped_error = 0
    for file_path in file_list:
        try:
            coords, is_down = load_delbot_trajectory(file_path)
            if len(coords) < max_len:
                skipped_short += 1
                continue
                                        
            coords = coords[:max_len]
            is_down = is_down[:max_len]
                          
            coords_norm = normalize_coords(coords)
                                                                              
            seq = np.hstack((coords_norm, is_down.reshape(-1, 1).astype(np.float32)))
            sequences.append(seq)
            labels.append(label)
        except Exception as e:
            skipped_error += 1
            continue
    print(f"   (файлів {label}: пропущено через коротку довжину (<{max_len} точок) — "
          f"{skipped_short}, пропущено через помилку читання — {skipped_error})")
    return np.array(sequences), np.array(labels)

                       
human_files = glob.glob(f'{DELBOT_PATH}/circles_human_*/*.csv')
if not human_files:
    human_files = glob.glob(f'{DELBOT_PATH}/circles_human_*/*')
bot_files = glob.glob(f'{DELBOT_PATH}/circles_bot_*/*.csv')
if not bot_files:
    bot_files = glob.glob(f'{DELBOT_PATH}/circles_bot_*/*')

print(f"Знайдено людей: {len(human_files)}, ботів: {len(bot_files)}")

                                   
print("Завантаження людських траєкторій...")
X_human, y_human = load_trajectories(human_files, 'human')
print(f"Людей: {len(X_human)}")

print("Завантаження бот-траєкторій...")
X_bot, y_bot = load_trajectories(bot_files, 'bot')
print(f"Ботів: {len(X_bot)}")

                      
X = np.concatenate([X_human, X_bot], axis=0)
y_str = np.concatenate([y_human, y_bot], axis=0)

print(f"Загальна кількість послідовностей: {len(X)}")
print(f"Форма X: {X.shape}")                           

                                                      
y = np.array([1 if label == 'bot' else 0 for label in y_str], dtype=np.float32)

                                    
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

                                                                          
n_bad = (~np.isfinite(X_train)).any(axis=(1, 2)).sum()
print(f"Послідовностей з NaN/Inf у X_train: {n_bad} з {len(X_train)}")
if n_bad > 0:
    mask_ok = np.isfinite(X_train).all(axis=(1, 2))
    X_train, y_train = X_train[mask_ok], y_train[mask_ok]
    print(f"Прибрано {n_bad} биту(их) послідовність(ей), лишилось {len(X_train)}")

                                
model = Sequential([
    LSTM(64, input_shape=(MAX_LEN, 3), return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

                                     
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

                           
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

                                                                      
y_test_str = ['bot' if v==1 else 'human' for v in y_test]
y_pred_str = ['bot' if v==1 else 'human' for v in y_pred]

print("\n" + "="*50)
print("ЗВІТ КЛАСИФІКАЦІЇ (LSTM на сирих послідовностях)")
print("="*50)
print(classification_report(y_test_str, y_pred_str))
print("\nМатриця плутанини:")
print(confusion_matrix(y_test_str, y_pred_str))

accuracy_lstm = (y_pred.flatten() == y_test).mean()
print(f"\nТочність (Accuracy) LSTM: {accuracy_lstm:.4f}")

                             
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
model.save(f'{PROJECT_DIR}/models/lstm_behavioral.h5')
print(f"\n✅ LSTM модель збережено у {PROJECT_DIR}/models/lstm_behavioral.h5")

                            
del X, y, X_train, X_test, y_train, y_test, X_human, X_bot, y_human, y_bot, y_str
gc.collect()

Знайдено людей: 955, ботів: 2596
Завантаження людських траєкторій...
   (файлів human: пропущено через коротку довжину (<100 точок) — 521, пропущено через помилку читання — 0)
Людей: 434
Завантаження бот-траєкторій...
   (файлів bot: пропущено через коротку довжину (<100 точок) — 2200, пропущено через помилку читання — 0)
Ботів: 396
Загальна кількість послідовностей: 830
Форма X: (830, 100, 3)
Послідовностей з NaN/Inf у X_train: 0 з 664


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        17,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,521 (76.25 KB)

 Trainable params: 19,521 (76.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 66ms/step - accuracy: 0.4915 - loss: 0.7554 - val_accuracy: 0.6767 - val_loss: 0.6685
Epoch 2/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.5593 - loss: 0.6851 - val_accuracy: 0.6241 - val_loss: 0.6511
Epoch 3/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.5650 - loss: 0.6835 - val_accuracy: 0.6541 - val_loss: 0.6401
Epoch 4/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.5650 - loss: 0.6704 - val_accuracy: 0.6541 - val_loss: 0.6391
Epoch 5/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.5782 - loss: 0.6805 - val_accuracy: 0.6767 - val_loss: 0.6314
Epoch 6/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.6045 - loss: 0.6581 - val_accuracy: 0.6842 - val_loss: 0.6226
Epoch 7/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.6139 - loss: 0.6435 - val_accuracy: 0.6767 - val_loss: 0.6200
Epoch 8/50
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.6422 - loss: 0.6316 - val_accuracy: 0.6842 - val_loss: 0.6116



ЗВІТ КЛАСИФІКАЦІЇ (LSTM на сирих послідовностях)
              precision    recall  f1-score   support

         bot       0.79      0.63      0.70        79
       human       0.72      0.85      0.78        87

    accuracy                           0.75       166
   macro avg       0.76      0.74      0.74       166
weighted avg       0.75      0.75      0.74       166


Матриця плутанини:
[[50 29]
 [13 74]]

Точність (Accuracy) LSTM: 0.7470

✅ LSTM модель збережено у /content/drive/MyDrive/man-bot-detection/models/lstm_behavioral.h5


7936

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

                              
PROJECT_DIR = '/content/drive/MyDrive/man-bot-detection'
df = pd.read_csv(f'{PROJECT_DIR}/data/processed/behavioral_features_windowed.csv')

                                                  
X = df.drop(['label', 'source'], axis=1)
y = df['label']
source = df['source']

                                                          
X_train, X_test, y_train, y_test, source_train, source_test = train_test_split(
    X, y, source, test_size=0.2, stratify=y, random_state=42
)

                                           
rf = joblib.load(f'{PROJECT_DIR}/models/behavioral_rf_300.pkl')

                         
y_pred = rf.predict(X_test)

                                     
                                                
unique_sources = source_test.unique()
print("="*60)
print("ОЦІНКА МОДЕЛІ ЗА ДЖЕРЕЛАМИ ДАНИХ")
print("="*60)

                                    
results = []

for src in sorted(unique_sources):
                                                   
    mask = (source_test == src)
    y_true_src = y_test[mask]
    y_pred_src = y_pred[mask]

                                                        
    if len(y_true_src) == 0:
        continue

                        
    acc = accuracy_score(y_true_src, y_pred_src)
    report = classification_report(y_true_src, y_pred_src, output_dict=True, zero_division=0)

                                                              
                                              
    bot_metrics = report.get('bot', {'precision': 0, 'recall': 0, 'f1-score': 0, 'support': 0})
    human_metrics = report.get('human', {'precision': 0, 'recall': 0, 'f1-score': 0, 'support': 0})

    results.append({
        'source': src,
        'accuracy': acc,
        'bot_precision': bot_metrics['precision'],
        'bot_recall': bot_metrics['recall'],
        'bot_f1': bot_metrics['f1-score'],
        'bot_support': bot_metrics['support'],
        'human_precision': human_metrics['precision'],
        'human_recall': human_metrics['recall'],
        'human_f1': human_metrics['f1-score'],
        'human_support': human_metrics['support']
    })

                                     
results_df = pd.DataFrame(results)

                                                               
                                                                           
                                                                          
                                   
bot_sources = results_df[results_df['source'].str.endswith('_bot')]
human_sources = results_df[results_df['source'].str.endswith('_human')]

print("\n🔹 ДЖЕРЕЛА БОТІВ:")
print(bot_sources[['source', 'bot_precision', 'bot_recall', 'bot_f1', 'bot_support']].to_string(index=False))

print("\n🔹 ДЖЕРЕЛА ЛЮДЕЙ (для порівняння):")
print(human_sources[['source', 'human_precision', 'human_recall', 'human_f1', 'human_support']].to_string(index=False))

                                                  
                                                                          
                                                                     
mask_bot = (y_test == 'bot')
y_true_bot = y_test[mask_bot]
y_pred_bot = y_pred[mask_bot]
print("\n ЗАГАЛЬНИЙ ЗВІТ ДЛЯ ВСІХ БОТІВ (разом):")
print(classification_report(y_true_bot, y_pred_bot, zero_division=0))

                                                                 
                                                                       
                                                                         
                        
print("\n🔹 ОКРЕМО ПО КОЖНОМУ ЛЮДСЬКОМУ ДЖЕРЕЛУ:")
for src in sorted(human_sources['source'].unique()):
    mask_src = (source_test == src)
    print(f"\n--- {src} ---")
    print(classification_report(y_test[mask_src], y_pred[mask_src], zero_division=0))

ОЦІНКА МОДЕЛІ ЗА ДЖЕРЕЛАМИ ДАНИХ

🔹 ДЖЕРЕЛА БОТІВ:
          source  bot_precision  bot_recall   bot_f1  bot_support
delbot_mouse_bot            1.0    0.827804 0.905791        633.0

🔹 ДЖЕРЕЛА ЛЮДЕЙ (для порівняння):
               source  human_precision  human_recall  human_f1  human_support
captchasolve30k_human              1.0      0.995411  0.997700         6973.0
   delbot_mouse_human              1.0      0.970711  0.985138          239.0

 ЗАГАЛЬНИЙ ЗВІТ ДЛЯ ВСІХ БОТІВ (разом):
              precision    recall  f1-score   support

         bot       1.00      0.83      0.91       633
       human       0.00      0.00      0.00         0

    accuracy                           0.83       633
   macro avg       0.50      0.41      0.45       633
weighted avg       1.00      0.83      0.91       633


🔹 ОКРЕМО ПО КОЖНОМУ ЛЮДСЬКОМУ ДЖЕРЕЛУ:

--- captchasolve30k_human ---
              precision    recall  f1-score   support

         bot       0.00      0.00      0.00         0


In [ ]:
import os

PROJECT_DIR = '/content/drive/MyDrive/man-bot-detection'

files_to_check = {
    "Ознаки (windowed CSV, DELBOT+Captcha)": f"{PROJECT_DIR}/data/processed/behavioral_features_windowed.csv",
    "RandomForest модель": f"{PROJECT_DIR}/models/behavioral_rf_300.pkl",
    "LSTM модель": f"{PROJECT_DIR}/models/lstm_behavioral.h5",
    "Кеш CaptchaSolve30k": f"{PROJECT_DIR}/data/raw/captchasolve30k_hf_cache",
}

print("="*60)
print("ПІДСУМОК СЕСІЇ — що збережено на Google Drive")
print("="*60)

for name, path in files_to_check.items():
    if os.path.isfile(path):
        size = os.path.getsize(path) / 1024 / 1024
        print(f"✅ {name}\n   {path}\n   Розмір: {size:.2f} МБ\n")
    elif os.path.isdir(path):
        n = len(os.listdir(path))
        print(f"✅ {name}\n   {path}\n   Файлів усередині: {n}\n")
    else:
        print(f"❌ {name} — НЕ ЗНАЙДЕНО за шляхом {path}\n")

print("Усе, що позначено ✅, залишиться на Drive і завтра не потребує")
print("повторного перерахунку — просто запусти комірку 'ВІДНОВЛЕННЯ' нижче.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import joblib
from tensorflow.keras.models import load_model
from sklearn.model_selection import train_test_split

PROJECT_DIR = '/content/drive/MyDrive/man-bot-detection'

                                                                                     
df = pd.read_csv(f'{PROJECT_DIR}/data/processed/behavioral_features_windowed.csv')
print(f"✅ Завантажено ознаки: {df.shape[0]} рядків, {df.shape[1]} колонок")
print(df['label'].value_counts())
print(df['source'].value_counts())

                                                                              
X = df.drop(['label', 'source'], axis=1)
y = df['label']
source = df['source']

X_train, X_test, y_train, y_test, source_train, source_test = train_test_split(
    X, y, source, test_size=0.2, stratify=y, random_state=42
)
print(f"\n✅ Спліт відтворено: train={X_train.shape[0]}, test={X_test.shape[0]}")

                                                                             
rf = joblib.load(f'{PROJECT_DIR}/models/behavioral_rf_300.pkl')
print(f"✅ RandomForest модель завантажена: {rf.n_estimators} дерев")

                                                                     
lstm_model = load_model(f'{PROJECT_DIR}/models/lstm_behavioral.h5')
print(f"✅ LSTM модель завантажена")

print("\n🎯 Усе готово для продовження — df, X_train/X_test/y_train/y_test, "
      "rf, lstm_model доступні в пам'яті. Можеш одразу перейти до аналізу "
      "результатів або до 03_network_model.ipynb.")